# Text Classification: BiLSTM NLP Model & Experiment Tracking

Notebook ini berisi langkah-langkah untuk:
1. **Deep Learning Modelling**: Melatih model **Bidirectional LSTM** dengan Keras Embedding layer.
2. **Hyperparameter Tweaking**: Menjalankan 4 konfigurasi eksperimen berbeda yang di-log ke MLflow.
3. **Model Terbaik**: Membandingkan hasil eksperimen dan menampilkan model terbaik berdasarkan F1-Score.

Data yang digunakan adalah data hasil pembersihan dari tahap sebelumnya.
Untuk mengubah konfigurasi eksperimen, edit nilai di `EXPERIMENTS_CONFIG`.

In [1]:
import pandas as pd
import numpy as np
import time
import os
import warnings
import urllib3

urllib3.disable_warnings()
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Embedding, Bidirectional, LSTM, Dense, Dropout, SpatialDropout1D
)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, classification_report
)

import mlflow
import mlflow.tensorflow

print("TensorFlow version:", tf.__version__)
print("MLflow version:", mlflow.__version__)

/Users/enricko/Kuliah/Data Mining/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


TensorFlow version: 2.16.2
MLflow version: 3.1.4


### 1. Load Dataset & Preprocessing (Shared Across All Experiments)

In [2]:
df = pd.read_csv('results/data/dataset_cleaned.csv')
df = df.dropna(subset=['cleaned_text', 'sentiment'])

print("Total data:", len(df))
print("\nDistribusi Kelas:")
print(df['sentiment'].value_counts())

# Label Encoding
le = LabelEncoder()
y = le.fit_transform(df['sentiment'])
num_classes = len(le.classes_)
print(f"\nJumlah kelas: {num_classes}")
print("Kelas:", le.classes_.tolist())

Total data: 59591

Distribusi Kelas:
sentiment
happiness     11942
sadness       10944
worry         10782
neutral        8433
love           5431
surprise       2892
anger          2819
fun            1774
relief         1521
hate           1318
empty           797
enthusiasm      759
boredom         179
Name: count, dtype: int64

Jumlah kelas: 13
Kelas: ['anger', 'boredom', 'empty', 'enthusiasm', 'fun', 'happiness', 'hate', 'love', 'neutral', 'relief', 'sadness', 'surprise', 'worry']


In [3]:
# Train-Test Split (dilakukan SEKALI, dipakai semua eksperimen)
X_raw = df['cleaned_text'].astype(str).values

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {len(X_train_raw)}, Test: {len(X_test_raw)}")

Train: 47672, Test: 11919


In [4]:
# Class Weights (menggantikan SMOTE — lebih tepat untuk deep learning)
classes = np.unique(y_train)
weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weight_dict = dict(zip(classes.tolist(), weights.tolist()))

print("Class Weights:")
for idx, name in enumerate(le.classes_):
    print(f"  {name}: {class_weight_dict[idx]:.4f}")

Class Weights:
  anger: 1.6262
  boredom: 25.6439
  empty: 5.7478
  enthusiasm: 6.0413
  fun: 2.5843
  happiness: 0.3839
  hate: 3.4792
  love: 0.8440
  neutral: 0.5436
  relief: 3.0132
  sadness: 0.4189
  surprise: 1.5847
  worry: 0.4251


### 2. Konfigurasi Eksperimen (PARAMS)

Edit nilai di `EXPERIMENTS_CONFIG` untuk mencoba hyperparameter yang berbeda.
Setiap entry akan dijalankan sebagai satu MLflow run terpisah.

| Parameter | Keterangan |
|---|---|
| `vocab_size` | Jumlah kata teratas yang dipakai sebagai fitur |
| `max_len` | Panjang maksimum urutan token (padding/truncate) |
| `embedding_dim` | Dimensi vektor embedding per token |
| `lstm_units` | Jumlah unit LSTM (×2 karena Bidirectional) |
| `dropout_rate` | Kekuatan regularisasi dropout (0.0–1.0) |
| `learning_rate` | Learning rate optimizer Adam |
| `batch_size` | Jumlah sampel per batch training |
| `epochs` | Maksimum epoch (EarlyStopping bisa berhenti lebih awal) |

In [5]:
EXPERIMENTS_CONFIG = [
    {
        "run_name": "BiLSTM_Baseline",
        "PARAMS": {
            "vocab_size":    10000,
            "max_len":          30,
            "embedding_dim":    64,
            "lstm_units":       64,
            "dropout_rate":    0.3,
            "learning_rate": 0.001,
            "batch_size":       64,
            "epochs":           20,
        }
    },
    {
        "run_name": "BiLSTM_LargerVocab",
        "PARAMS": {
            "vocab_size":    20000,
            "max_len":          30,
            "embedding_dim":   128,
            "lstm_units":      128,
            "dropout_rate":    0.3,
            "learning_rate": 0.001,
            "batch_size":       64,
            "epochs":           20,
        }
    },
    {
        "run_name": "BiLSTM_DeepDropout",
        "PARAMS": {
            "vocab_size":    20000,
            "max_len":          30,
            "embedding_dim":   128,
            "lstm_units":      128,
            "dropout_rate":    0.5,
            "learning_rate": 0.0005,
            "batch_size":       32,
            "epochs":           25,
        }
    },
    {
        "run_name": "BiLSTM_StackedLSTM",
        "PARAMS": {
            "vocab_size":    20000,
            "max_len":          30,
            "embedding_dim":   128,
            "lstm_units":      128,
            "dropout_rate":    0.4,
            "learning_rate": 0.001,
            "batch_size":       64,
            "epochs":           20,
        }
    },
]

### 3. Helper Functions: Tokenizer, Sequences, dan Model Builder

In [6]:
def build_tokenizer_and_sequences(X_train_raw, X_test_raw, vocab_size, max_len):
    """Fit tokenizer on train only, then transform both splits."""
    tokenizer = Tokenizer(num_words=vocab_size, oov_token='<OOV>')
    tokenizer.fit_on_texts(X_train_raw)

    X_train_pad = pad_sequences(
        tokenizer.texts_to_sequences(X_train_raw),
        maxlen=max_len, padding='post', truncating='post'
    )
    X_test_pad = pad_sequences(
        tokenizer.texts_to_sequences(X_test_raw),
        maxlen=max_len, padding='post', truncating='post'
    )
    return tokenizer, X_train_pad, X_test_pad


def build_bilstm_model(vocab_size, max_len, embedding_dim, lstm_units,
                       dropout_rate, learning_rate, num_classes, stacked=False):
    """Build a Bidirectional LSTM model. stacked=True adds a second BiLSTM layer."""
    model = Sequential()
    # vocab_size + 1: Tokenizer indices run 0 (pad) to vocab_size inclusive
    model.add(Embedding(input_dim=vocab_size + 1,
                        output_dim=embedding_dim,
                        input_length=max_len))
    # SpatialDropout1D drops entire embedding dimensions — better than Dropout for sequences
    model.add(SpatialDropout1D(dropout_rate))

    if stacked:
        model.add(Bidirectional(LSTM(lstm_units, return_sequences=True, dropout=dropout_rate)))
        model.add(Bidirectional(LSTM(lstm_units // 2, dropout=dropout_rate)))
    else:
        model.add(Bidirectional(LSTM(lstm_units, dropout=dropout_rate)))

    model.add(Dense(64, activation='relu'))
    model.add(Dropout(dropout_rate))
    model.add(Dense(num_classes, activation='softmax'))

    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


print("Helper functions siap.")

Helper functions siap.


### 4. Jalankan Eksperimen & Log ke MLflow

In [7]:
mlflow.set_experiment("Emotion_Classification_Experiment")

experiment_logs = []

for config in EXPERIMENTS_CONFIG:
    run_name = config["run_name"]
    P = config["PARAMS"]
    is_stacked = (run_name == "BiLSTM_StackedLSTM")

    print(f"\n{'='*60}")
    print(f"Menjalankan: {run_name}")
    print(f"PARAMS: {P}")
    print(f"{'='*60}")

    # Bangun tokenizer & sequence baru per config (vocab_size bisa berbeda)
    _, X_train_pad, X_test_pad = build_tokenizer_and_sequences(
        X_train_raw, X_test_raw,
        vocab_size=P['vocab_size'],
        max_len=P['max_len']
    )

    # Bangun model
    model = build_bilstm_model(
        vocab_size=P['vocab_size'],
        max_len=P['max_len'],
        embedding_dim=P['embedding_dim'],
        lstm_units=P['lstm_units'],
        dropout_rate=P['dropout_rate'],
        learning_rate=P['learning_rate'],
        num_classes=num_classes,
        stacked=is_stacked
    )

    early_stop = EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True,
        verbose=0
    )

    with mlflow.start_run(run_name=run_name):
        # Log semua hyperparameter
        mlflow.log_param("model_type", "BiLSTM")
        mlflow.log_param("stacked", is_stacked)
        for key, val in P.items():
            mlflow.log_param(key, val)

        # Training
        start_time = time.time()
        history = model.fit(
            X_train_pad, y_train,
            epochs=P['epochs'],
            batch_size=P['batch_size'],
            validation_split=0.1,
            class_weight=class_weight_dict,
            callbacks=[early_stop],
            verbose=1
        )
        training_time = time.time() - start_time
        epochs_ran = len(history.history['loss'])

        # Prediksi
        y_pred = np.argmax(model.predict(X_test_pad, verbose=0), axis=1)

        # Metrik
        acc  = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
        rec  = recall_score(y_test, y_pred, average='weighted', zero_division=0)
        f1   = f1_score(y_test, y_pred, average='weighted', zero_division=0)

        # Log metrik ke MLflow
        mlflow.log_metric("training_time", round(training_time, 4))
        mlflow.log_metric("accuracy",      round(acc,  4))
        mlflow.log_metric("precision",     round(prec, 4))
        mlflow.log_metric("recall",        round(rec,  4))
        mlflow.log_metric("f1_score",      round(f1,   4))

        # Log model
        mlflow.tensorflow.log_model(model, name="bilstm_model")

        print(f"\n{run_name} selesai.")
        print(f"  Epochs: {epochs_ran}/{P['epochs']} | F1-Score: {f1:.4f} | Accuracy: {acc:.4f} | Time: {training_time:.1f}s")

        experiment_logs.append({
            'Run Name':          run_name,
            'vocab_size':        P['vocab_size'],
            'max_len':           P['max_len'],
            'embedding_dim':     P['embedding_dim'],
            'lstm_units':        P['lstm_units'],
            'dropout_rate':      P['dropout_rate'],
            'learning_rate':     P['learning_rate'],
            'batch_size':        P['batch_size'],
            'epochs_ran':        epochs_ran,
            'Training Time (s)': round(training_time, 2),
            'Accuracy':          round(acc,  4),
            'Precision':         round(prec, 4),
            'Recall':            round(rec,  4),
            'F1-Score':          round(f1,   4),
        })

print("\nSemua eksperimen selesai.")


Menjalankan: BiLSTM_Baseline
PARAMS: {'vocab_size': 10000, 'max_len': 30, 'embedding_dim': 64, 'lstm_units': 64, 'dropout_rate': 0.3, 'learning_rate': 0.001, 'batch_size': 64, 'epochs': 20}
Epoch 1/20
671/671 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - accuracy: 0.0928 - loss: 2.4526 - val_accuracy: 0.3112 - val_loss: 2.1466
Epoch 2/20
671/671 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.2632 - loss: 2.0962 - val_accuracy: 0.3748 - val_loss: 1.8778
Epoch 3/20
671/671 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.3837 - loss: 1.8727 - val_accuracy: 0.3911 - val_loss: 1.8137
Epoch 4/20
671/671 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.4212 - loss: 1.7090 - val_accuracy: 0.4027 - val_loss: 1.7448
Epoch 5/20
671/671 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.4496 - loss: 1.5999 - val_accuracy: 0.3966 - val_loss: 1.7697
Epoch 6/20
671/671 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.4654 - loss: 1.4896 - val_accuracy: 0.4096 - val_loss: 1.7273
Epoch 7/20
671/671 ━━━━━━━━━━━━━━━━━━━━ 5

2026/05/01 18:28:25 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.
2026/05/01 18:28:29 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.



BiLSTM_Baseline selesai.
  Epochs: 9/20 | F1-Score: 0.4481 | Accuracy: 0.4105 | Time: 50.2s

Menjalankan: BiLSTM_LargerVocab
PARAMS: {'vocab_size': 20000, 'max_len': 30, 'embedding_dim': 128, 'lstm_units': 128, 'dropout_rate': 0.3, 'learning_rate': 0.001, 'batch_size': 64, 'epochs': 20}
Epoch 1/20
671/671 ━━━━━━━━━━━━━━━━━━━━ 13s 18ms/step - accuracy: 0.1014 - loss: 2.4644 - val_accuracy: 0.3408 - val_loss: 2.0103
Epoch 2/20
671/671 ━━━━━━━━━━━━━━━━━━━━ 12s 18ms/step - accuracy: 0.3239 - loss: 2.0300 - val_accuracy: 0.3897 - val_loss: 1.8006
Epoch 3/20
671/671 ━━━━━━━━━━━━━━━━━━━━ 12s 19ms/step - accuracy: 0.4204 - loss: 1.7791 - val_accuracy: 0.4283 - val_loss: 1.7384
Epoch 4/20
671/671 ━━━━━━━━━━━━━━━━━━━━ 12s 18ms/step - accuracy: 0.4841 - loss: 1.5112 - val_accuracy: 0.4287 - val_loss: 1.7462
Epoch 5/20
671/671 ━━━━━━━━━━━━━━━━━━━━ 12s 18ms/step - accuracy: 0.5104 - loss: 1.3609 - val_accuracy: 0.4247 - val_loss: 1.7253
Epoch 6/20
671/671 ━━━━━━━━━━━━━━━━━━━━ 12s 18ms/step - accur

2026/05/01 18:30:10 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.
2026/05/01 18:30:14 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.



BiLSTM_LargerVocab selesai.
  Epochs: 8/20 | F1-Score: 0.4691 | Accuracy: 0.4226 | Time: 98.3s

Menjalankan: BiLSTM_DeepDropout
PARAMS: {'vocab_size': 20000, 'max_len': 30, 'embedding_dim': 128, 'lstm_units': 128, 'dropout_rate': 0.5, 'learning_rate': 0.0005, 'batch_size': 32, 'epochs': 25}
Epoch 1/25
1341/1341 ━━━━━━━━━━━━━━━━━━━━ 19s 14ms/step - accuracy: 0.0933 - loss: 2.5352 - val_accuracy: 0.2085 - val_loss: 2.2622
Epoch 2/25
1341/1341 ━━━━━━━━━━━━━━━━━━━━ 19s 14ms/step - accuracy: 0.1802 - loss: 2.2823 - val_accuracy: 0.3781 - val_loss: 1.9784
Epoch 3/25
1341/1341 ━━━━━━━━━━━━━━━━━━━━ 19s 14ms/step - accuracy: 0.3107 - loss: 2.1069 - val_accuracy: 0.3586 - val_loss: 1.8606
Epoch 4/25
1341/1341 ━━━━━━━━━━━━━━━━━━━━ 18s 14ms/step - accuracy: 0.3546 - loss: 1.9408 - val_accuracy: 0.3737 - val_loss: 1.8181
Epoch 5/25
1341/1341 ━━━━━━━━━━━━━━━━━━━━ 18s 14ms/step - accuracy: 0.3846 - loss: 1.8928 - val_accuracy: 0.4151 - val_loss: 1.7550
Epoch 6/25
1341/1341 ━━━━━━━━━━━━━━━━━━━━ 18s 1

2026/05/01 18:33:45 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.
2026/05/01 18:33:49 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.



BiLSTM_DeepDropout selesai.
  Epochs: 11/25 | F1-Score: 0.4714 | Accuracy: 0.4250 | Time: 209.1s

Menjalankan: BiLSTM_StackedLSTM
PARAMS: {'vocab_size': 20000, 'max_len': 30, 'embedding_dim': 128, 'lstm_units': 128, 'dropout_rate': 0.4, 'learning_rate': 0.001, 'batch_size': 64, 'epochs': 20}
Epoch 1/20
671/671 ━━━━━━━━━━━━━━━━━━━━ 22s 30ms/step - accuracy: 0.0980 - loss: 2.4873 - val_accuracy: 0.0994 - val_loss: 2.3467
Epoch 2/20
671/671 ━━━━━━━━━━━━━━━━━━━━ 21s 32ms/step - accuracy: 0.1866 - loss: 2.2650 - val_accuracy: 0.3744 - val_loss: 1.8820
Epoch 3/20
671/671 ━━━━━━━━━━━━━━━━━━━━ 22s 32ms/step - accuracy: 0.3481 - loss: 1.9974 - val_accuracy: 0.3964 - val_loss: 1.8025
Epoch 4/20
671/671 ━━━━━━━━━━━━━━━━━━━━ 22s 32ms/step - accuracy: 0.4268 - loss: 1.7660 - val_accuracy: 0.4062 - val_loss: 1.7629
Epoch 5/20
671/671 ━━━━━━━━━━━━━━━━━━━━ 23s 34ms/step - accuracy: 0.4565 - loss: 1.6258 - val_accuracy: 0.4327 - val_loss: 1.7203
Epoch 6/20
671/671 ━━━━━━━━━━━━━━━━━━━━ 22s 33ms/step - 

2026/05/01 18:36:49 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.
2026/05/01 18:36:53 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.



BiLSTM_StackedLSTM selesai.
  Epochs: 8/20 | F1-Score: 0.4738 | Accuracy: 0.4307 | Time: 176.6s

Semua eksperimen selesai.


### 5. Evaluasi Eksperimen & Model Terbaik

In [8]:
# Tampilkan tabel perbandingan semua eksperimen
df_logs = pd.DataFrame(experiment_logs)
df_logs = df_logs.sort_values(by='F1-Score', ascending=False).reset_index(drop=True)

print("=== LOG SEMUA EKSPERIMEN ===")
display(df_logs)

# Simpan log ke CSV
os.makedirs('results', exist_ok=True)
df_logs.to_csv('results/experiment_logs.csv', index=False)
print("\nLog eksperimen disimpan ke 'results/experiment_logs.csv'")

=== LOG SEMUA EKSPERIMEN ===


,Run Name,vocab_size,max_len,embedding_dim,lstm_units,dropout_rate,learning_rate,batch_size,epochs_ran,Training Time (s),Accuracy,Precision,Recall,F1-Score
0,BiLSTM_StackedLSTM,20000,30,128,128,0.4,0.0010,64,8,176.56,0.4307,0.5689,0.4307,0.4738
1,BiLSTM_DeepDropout,20000,30,128,128,0.5,0.0005,32,11,209.07,0.4250,0.5643,0.4250,0.4714
2,BiLSTM_LargerVocab,20000,30,128,128,0.3,0.0010,64,8,98.35,0.4226,0.5618,0.4226,0.4691
3,BiLSTM_Baseline,10000,30,64,64,0.3,0.0010,64,9,50.25,0.4105,0.5584,0.4105,0.4481



Log eksperimen disimpan ke 'results/experiment_logs.csv'


In [9]:
# Model Terbaik — ringkasan
best_row = df_logs.iloc[0]

print("=" * 60)
print("MODEL TERBAIK")
print("=" * 60)
print(f"Run Name:         {best_row['Run Name']}")
print(f"F1-Score:         {best_row['F1-Score']}")
print(f"Accuracy:         {best_row['Accuracy']}")
print(f"Precision:        {best_row['Precision']}")
print(f"Recall:           {best_row['Recall']}")
print(f"Training Time:    {best_row['Training Time (s)']}s")
print(f"Epochs (actual):  {best_row['epochs_ran']}")
print("Hyperparameters:")
for col in ['vocab_size', 'max_len', 'embedding_dim', 'lstm_units',
            'dropout_rate', 'learning_rate', 'batch_size']:
    print(f"  {col}: {best_row[col]}")
print("=" * 60)

# Re-train model terbaik untuk classification report
best_run_name = best_row['Run Name']
best_config = next(c for c in EXPERIMENTS_CONFIG if c['run_name'] == best_run_name)
P_best = best_config['PARAMS']
is_stacked_best = (best_run_name == "BiLSTM_StackedLSTM")

_, X_train_best, X_test_best = build_tokenizer_and_sequences(
    X_train_raw, X_test_raw,
    vocab_size=P_best['vocab_size'],
    max_len=P_best['max_len']
)

best_model = build_bilstm_model(
    vocab_size=P_best['vocab_size'],
    max_len=P_best['max_len'],
    embedding_dim=P_best['embedding_dim'],
    lstm_units=P_best['lstm_units'],
    dropout_rate=P_best['dropout_rate'],
    learning_rate=P_best['learning_rate'],
    num_classes=num_classes,
    stacked=is_stacked_best
)

best_model.fit(
    X_train_best, y_train,
    epochs=P_best['epochs'],
    batch_size=P_best['batch_size'],
    validation_split=0.1,
    class_weight=class_weight_dict,
    callbacks=[EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=0)],
    verbose=0
)

y_pred_best = np.argmax(best_model.predict(X_test_best, verbose=0), axis=1)
print("\nClassification Report untuk Model Terbaik:")
print(classification_report(y_test, y_pred_best, target_names=le.classes_, zero_division=0))

MODEL TERBAIK
Run Name:         BiLSTM_StackedLSTM
F1-Score:         0.4738
Accuracy:         0.4307
Precision:        0.5689
Recall:           0.4307
Training Time:    176.56s
Epochs (actual):  8
Hyperparameters:
  vocab_size: 20000
  max_len: 30
  embedding_dim: 128
  lstm_units: 128
  dropout_rate: 0.4
  learning_rate: 0.001
  batch_size: 64

Classification Report untuk Model Terbaik:
              precision    recall  f1-score   support

       anger       0.82      0.92      0.87       564
     boredom       0.04      0.19      0.06        36
       empty       0.03      0.12      0.04       159
  enthusiasm       0.03      0.09      0.04       152
         fun       0.11      0.09      0.10       355
   happiness       0.80      0.50      0.61      2389
        hate       0.25      0.29      0.27       264
        love       0.40      0.61      0.48      1086
     neutral       0.35      0.33      0.34      1687
      relief       0.05      0.22      0.08       304
     sadness  

In [10]:
# Simpan model terbaik ke file untuk di-upload ke GitHub
model_save_path = f"results/best_model_{best_run_name}.keras"
best_model.save(model_save_path)
print(f"Model terbaik disimpan ke: {model_save_path}")
print(f"\nFile yang perlu di-push ke GitHub:")
print(f"  {model_save_path}")
print(f"  results/experiment_logs.csv")
print(f"  modeling_experiment.ipynb")

Model terbaik disimpan ke: results/best_model_BiLSTM_StackedLSTM.keras

File yang perlu di-push ke GitHub:
  results/best_model_BiLSTM_StackedLSTM.keras
  results/experiment_logs.csv
  modeling_experiment.ipynb


In [11]:
# Jalankan MLflow UI di background
print("MLflow UI berjalan di background.")
print("Akses di: http://127.0.0.1:5050")
get_ipython().system_raw('mlflow ui --port 5050 &')

MLflow UI berjalan di background.
Akses di: http://127.0.0.1:5050


/Users/enricko/Kuliah/Data Mining/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
[2026-05-01 18:39:32 +0700] [24205] [INFO] Starting gunicorn 23.0.0
[2026-05-01 18:39:32 +0700] [24205] [INFO] Listening at: http://127.0.0.1:5050 (24205)
[2026-05-01 18:39:32 +0700] [24205] [INFO] Using worker: sync
[2026-05-01 18:39:32 +0700] [24210] [INFO] Booting worker with pid: 24210
[2026-05-01 18:39:32 +0700] [24211] [INFO] Booting worker with pid: 24211
/Users/enricko/Kuliah/Data Mining/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
[2026-05-01 18:39:32 +0700] [24212] [INFO] Booting worker with pid: 242